In [51]:
from bps_scraper import UserInput, SchoolDetails, BPSScraperAgent
from huggingface_hub import InferenceClient
from pydantic import ValidationError
import json
import re

In [99]:
from huggingface_hub import InferenceClient

import os
from dotenv import load_dotenv

HF_TOKEN = os.getenv("HF_TOKEN")

In [3]:
student_data = {
    'grade': 1,
    'street_number': '50',
    'street_name': 'everett st',
    'zip_code': '02128'
}

# convert into UserInput
user_input = UserInput.model_validate(student_data)

# agent = BPSScraperAgent(user_input)
# school_list = agent.run()
# print("Scraped", len(school_list), "schools.")
# agent.close()

In [77]:
class SchoolChatbot:
    """
    This class is extra scaffolding around a model. It handles conversations with parents
    looking for school information for their children.
    
    Example usage:
        chatbot = SchoolChatbot()
        response = chatbot.get_response("What schools offer Spanish programs?")
    """
    
    def __init__(self):
        """
        Initialize the chatbot with a HF model ID
        """
        model_id = MY_MODEL if MY_MODEL else BASE_MODEL # define MY_MODEL in config.py if you create a new model in the HuggingFace Hub
        self.client = InferenceClient(model=model_id, token=HF_TOKEN)
        self.conversation_history = []
        
    def format_prompt(self, user_input):
        """
        Format the user's input into a proper prompt with context about the 
        Boston school system and available data.
        
        Args:
            user_input (str): The user's question about Boston schools
        
        Returns:
            str: A formatted prompt ready for the model
        """
        # Track conversation history
        self.conversation_history.append({"role": "user", "content": user_input})
        
        # System instructions with information about available school data
        system_instructions = """
        You are a helpful assistant that specializes in Boston Public Schools. You help parents find the right schools for their children.
        
        You have access to the following information about schools:
        - School name, address, contact information (email, website)
        - Distance from the user's home
        - School hours and preview dates
        - Before/after school programs
        - School description and focus
        - Grades offered
        - Eligibility zones and special application requirements
        - Quality tier rating
        - Uniform policy
        - Academic programs offered
        - Facility features
        - Student support services
        - Sports and community partners
        
        When helping users, collect relevant information like:
        1. Child's grade level
        2. Their address or neighborhood
        3. Specific program interests (languages, arts, sports, etc.)
        4. Special requirements (special education, after-school care, etc.)
        
        Once you have enough information, you'll query a database to find matching schools. You'll receive a list of information on different schools ordered by how far they are from the student's home, with the closest school coming first. Be conversational, helpful, and focused on the user's needs.
        """
        
        # Format the conversation history
        conversation = []
        for i, message in enumerate(self.conversation_history):
            if message["role"] == "user":
                conversation.append(f"User: {message['content']}")
            else:
                conversation.append(f"Assistant: {message['content']}")
        
        # Combine everything into the final prompt
        formatted_prompt = f"{system_instructions}\n\n{''.join(conversation)}\n\nAssistant:"
        
        return formatted_prompt
    
    def extract_json(self, text: str) -> str:
        """
        Attempts to extract a valid JSON substring from the given text.
        First, it looks for a code block between triple backticks; if not found,
        it falls back to scanning between the first '{' and last '}'.
        """
        # Try to extract JSON within triple backticks (optionally labeled as json)
        match = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
        if match:
            return match.group(1)
        else:
            # Fallback: find first occurrence of '{' and last occurrence of '}'
            start = text.find('{')
            end = text.rfind('}')
            if start != -1 and end != -1 and end > start:
                return text[start:end + 1]
            else:
                raise ValueError("No valid JSON found in the text.")
    
    def extract_query_parameters(self, user_input):
        """
        Extract parameters from user input that can be used to query the school database.
        
        Args:
            user_input (str): The user's question about Boston schools
            
        Returns:
            dict: Parameters that can be used to query the school database
        """
        # Build a prompt to extract structured information
        extraction_prompt = f"""
        Extract the following information from the user's message if present:
        - Child's grade level
        - Street number of home address
        - Street name of home address
        - Zip code of home address

        ONLY OUTPUT THE JSON. YOUR OUTPUT SHOULD BE VALID JSON AND NOTHING ELSE.

        EXAMPLE 1
        <|user|>
        I live on 50 everett st in zip code 02128. I'm looking for a school for my child who is 1st grade.
        
        <|assistant|>
        {{
            "grade": "1",
            "street_number": "50",
            "street_name": "everett st",
            "zip_code": "02128"
        }}

        EXAMPLE 2
        <|user|>
        My child Anthony is a junior in high school and we live on 123 maple st in zip code 02130. We're moving to Allston next month and need to find a school for him to attend.

        <|assistant|>
        {{
            "grade": "12",
            "street_number": "123",
            "street_name": "maple st",
            "zip_code": "02130"
        }}
        </END EXAMPLE 2>

        Now here is the user's message. ONLY OUTPUT THE JSON. YOUR OUTPUT SHOULD BE VALID JSON AND NOTHING ELSE.
    
        <|user|>
        "{user_input}"
        
        <|assistant|>
        """
        
        # Get parameters from the LLM
        response = self.client.chat_completion(
            messages=[{"role": "user", "content": extraction_prompt}],
            temperature=0.0,  # Use low temperature for deterministic output
            max_tokens=300,
        )

        response_text = response.choices[0].message.content

        print('response text', response_text)

        json_data = self.extract_json(response_text)
        print('json data', json_data)
        # Parse the JSON response

        try:
            user_input = UserInput.model_validate_json(json_data)
            # Optionally, output as JSON in a formatted way:
            print("Validated user input:")
            print(user_input.model_dump_json(indent=2))
        except ValidationError as e:
            print("Validation error:", e.json())
        return user_input
        
    
    def query_school_database(self, parameters):
        """
        Mock function to represent querying the school database.
        In a real implementation, this would use the parameters to query your actual database.
        
        Args:
            parameters (dict): The extracted parameters from user input
            
        Returns:
            list: List of matching schools
        """
        # This is where you would integrate with your BPSScraperAgent
        # For example:
        # student_data = {
        #     'grade': parameters['grade_level'] or 1,
        #     'street_number': '50',  # You'd extract this from parameters['location']
        #     'street_name': 'everett st',  # You'd extract this from parameters['location']
        #     'zip_code': parameters['zip_code'] or '02128'
        # }
        # agent = BPSScraperAgent(student_data)
        # school_list = agent.run()
        # return school_list

        # convert into UserInput
        agent = BPSScraperAgent(parameters)
        school_list = agent.run()
        print("Scraped", len(school_list), "schools.")
        agent.close()
        
        # For now, just return a mock response
        return school_list
    
    def format_school_information(self, schools_list):
        """
        Format a list of SchoolDetails objects into a readable, well-structured string
        that can be used as context for the language model.
        
        Args:
            schools_list (list): List of SchoolDetails pydantic objects
            
        Returns:
            str: Formatted school information
        """
        if not schools_list:
            return "No schools found matching your criteria."
        
        formatted_output = "# MATCHING SCHOOLS INFORMATION\n\n"
        
        for i, school in enumerate(schools_list, 1):
            # Main header with school name and key info
            formatted_output += f"## {i}. {school.school_name}\n"
            formatted_output += f"**Distance:** {school.distance_from_home} | **Grades:** {school.grades_offered}\n\n"
            
            # Basic contact information
            formatted_output += "### Contact & Location\n"
            formatted_output += f"- **Address:** {school.address}\n"
            if school.school_website:
                formatted_output += f"- **Website:** {school.school_website}\n"
            if school.email:
                formatted_output += f"- **Email:** {school.email}\n"
            
            # Hours and scheduling
            if school.school_hours:
                formatted_output += f"- **Hours:** {school.school_hours}\n"
            formatted_output += "\n"
            
            # School profile
            formatted_output += "### School Profile\n"
            if school.school_description:
                formatted_output += f"- **Description:** {school.school_description.strip()}\n"
            if school.school_focus:
                formatted_output += f"- **Focus:** {school.school_focus.strip()}\n"
            if school.programs:
                formatted_output += f"- **Programs:** {school.programs}\n"
            if school.quality:
                formatted_output += f"- **Quality:** {school.quality}\n"
            formatted_output += "\n"
            
            # Admissions & eligibility
            formatted_output += "### Admissions Information\n"
            formatted_output += f"- **Eligibility:** {school.eligibility}\n"
            formatted_output += f"- **Special Application Required:** {school.special_application}\n"
            if school.preview_dates:
                formatted_output += f"- **Preview Dates:** {school.preview_dates}\n"
            if school.demand_reports:
                demand_info = school.demand_reports.replace("\n", " ").strip()
                formatted_output += f"- **Demand:** {demand_info}\n"
            formatted_output += "\n"
            
            # Additional services and facilities 
            formatted_output += "### Additional Features\n"
            if school.surround_care:
                formatted_output += f"- **Before/After School Programs:** {school.surround_care}\n"
            if school.facility_features:
                formatted_output += f"- **Facilities:** {school.facility_features}\n"
            if school.student_support:
                formatted_output += f"- **Student Support:** {school.student_support}\n"
            if school.sports:
                formatted_output += f"- **Sports:** {school.sports}\n"
            if school.community_partners:
                formatted_output += f"- **Community Partners:** {school.community_partners}\n"
            if school.uniform_policy:
                formatted_output += f"- **Uniform Policy:** {school.uniform_policy}\n"
            
            # Add separator between schools
            formatted_output += "\n" + "-" * 80 + "\n\n"
        
        return formatted_output

    
    def format_school_results(self, schools, user_preferences):
        """
        Format the school results for the chatbot response.
        
        Args:
            schools (list): List of matching schools
            user_preferences (dict): User preferences extracted from conversation
            
        Returns:
            str: Formatted school results
        """
        # Build a prompt to format the results in a helpful way
        format_prompt = f"""
        <|system|>
        You have a list of 3-10 schools that match the user's preferences.
        You have access to the following information for each school:
        - address: The physical address of the school
        - school_website: The website of the school
        - email: The email of the school
        - distance_from_home: The distance from the user's home to the school in miles
        - school_hours: The hours of the school
        - preview_dates: The dates of the school's preview dates for prospective students
        - surround_care: The surround care of the school, like before/after school programs
        - school_description: A description of the school
        - grades_offered: The grades offered by the school
        - eligibility: The eligibility of the school
        - special_application: The special application of the school
        - quality: The quality of the school
        - uniform_policy: The uniform policy of the school
        - school_focus: The focus of the school
        - programs: The programs offered by the school
        - facility_features: The facility features of the school
        - student_support: The student support services of the school
        - sports: The sports offered by the school
        - community_partners: The community partners of the school, like local businesses or organizations that the school is partnered with

        Format the following school information in a helpful, conversational response. Provide the first school's information first, then the second, etc.
        User preferences:
        {user_preferences}
        
        School information (in a list):
        {self.format_school_information(schools)}
        
        Focus on what's most relevant to the user's needs. Organize the information well and highlight key details that match their preferences.
        """
        
        # Get formatting from the LLM
        response = self.client.chat_completion(
            messages=[{"role": "user", "content": format_prompt}],
            temperature=0.7,
            max_tokens=2000,
        )
        
        return format_prompt, response.choices[0].message.content
    
    def get_response(self, user_input):
        """
        Generate responses to user questions about Boston schools.
        
        Args:
            user_input (str): The user's question about Boston schools
            
        Returns:
            str: The chatbot's response
        """
        # Format the prompt with conversation history
        prompt = self.format_prompt(user_input)
        
        # For advanced functionality, extract structured data from user input
        query_parameters = self.extract_query_parameters(user_input)
        
        # If we have enough information to query the database
        has_location = query_parameters.get('location') or query_parameters.get('zip_code')
        
        if has_location:
            # Query the database for matching schools
            matching_schools = self.query_school_database(query_parameters)
            
            if matching_schools:
                # Format the results
                response_content = self.format_school_results(matching_schools, query_parameters)
            else:
                # No matching schools found
                response = self.client.chat_completion(
                    messages=[
                        {"role": "user", "content": f"{prompt}\n\nI searched our database but couldn't find any schools matching those criteria. Let me ask for more information or suggest alternatives."}
                    ],
                    temperature=0.7,
                    max_tokens=500,
                )
                response_content = response.choices[0].message.content
        else:
            # Not enough information yet, just continue the conversation
            response = self.client.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=500,
            )
            response_content = response.choices[0].message.content
        
        # Store the response in conversation history
        self.conversation_history.append({"role": "assistant", "content": response_content})
        
        return response_content

    def reset_conversation(self):
        """Reset the conversation history"""
        self.conversation_history = []

In [17]:
import torch
from huggingface_hub import login

In [6]:
login()

In [126]:
# BASE_MODEL = "google/gemma-2-2b-it"
# BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MY_MODEL = None

In [127]:
chatbot = SchoolChatbot()

In [130]:
user_input = chatbot.extract_query_parameters("We live on 15 Hancock St, Boston MA 02114. My child will be in 1st grade")

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/hf-inference/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/v1/chat/completions (Request ID: Root=1-6804a35f-07647fdc12364da900b6fae3;a9780a77-c079-4f0e-b8d3-381d43719661)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [46]:
school_list = chatbot.query_school_database(user_input)

Page is fully loaded.
Page is fully loaded.
Clicked button 'home_search_button' (attempt 1).
Target element 'addresses_next_button' found after 1 attempts.
Clicked button 'addresses_next_button' (attempt 1).
Button 'addresses_next_button' not found. It might not be available on this page.
Clicked button 'ell_next_button' (attempt 1).
Target element 'sped_next_button' found after 1 attempts.
Clicked button 'sped_next_button' (attempt 1).
Target element 'home_schools_list' found after 1 attempts.
Success for Eliot K-8 Innovation School
Success for Quincy Elementary School
Success for Harvard-Kent Elementary School
Success for Warren-Prescott K-8 School
Success for Blackstone Elementary School
Success for Adams Elementary School
Success for Condon K-8 School
Success for East Boston Early Education Center
Success for Dudley Street Neighborhood School
Success for Hernandez Elementary School
Success for UP Academy Dorchester
Scraped 11 schools.


In [123]:
print(chatbot.format_school_information(school_list[:3]))

# MATCHING SCHOOLS INFORMATION

## 1. Eliot K-8 Innovation School
**Distance:** 0.862 mi from home | **Grades:** K0 - 8

### Contact & Location
- **Address:** 16 Charter St Boston MA 02113
- **Website:** http://bostonpublicschools.org/Page/628
- **Email:** eliot@bostonpublicschools.org
- **Hours:** 8:30am - 3:30pm

### School Profile
- **Description:** We provide an inclusive, joyful learning journey preparing students to achieve their highest potential by embracing their identities, developing interdisciplinary 21st century skills, empowered by knowledge to participate actively in a complex and constantly changing, culturally diverse world.
- **Focus:** We provide an inclusive, joyful, rigorous learning journey, preparing students to achieve their highest potential by embracing their identities, developing interdisciplinary skills, and applying an antiracist mindset to participate actively in the world.
- **Programs:** Arts, Inclusion, Phys Education
- **Quality:** BPS Quality Tier - 

In [129]:
user_input = "i'm looking for a school that's close to our home and safe, with good after school programs and sports"
prompt, school_recs = chatbot.format_school_results(school_list[:3], user_input)

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/hf-inference/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/v1/chat/completions (Request ID: Root=1-6804a354-4ee664ea202fee8031e1c595;a427ac33-ad74-44c2-a54c-ac9b0b7b84ce)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [118]:
from huggingface_hub import model_info

In [120]:
info = model_info(BASE_MODEL)

In [121]:
info.inference

'warm'

In [2]:
student_data = {
    'grade': 1,
    'street_number': '50',
    'street_name': 'everett st',
    'zip_code': '02128'
}

agent = BPSScraperAgent(student_data)
school_list = agent.run()
print("Scraped", len(school_list), "schools.")
agent.close()

Page is fully loaded.
Page is fully loaded.
Clicked button 'home_search_button' (attempt 1).
Clicked button 'home_search_button' (attempt 2).
Clicked button 'home_search_button' (attempt 3).
Target element 'addresses_next_button' found after 3 attempts.
Clicked button 'addresses_next_button' (attempt 1).
Clicked button 'addresses_next_button' (attempt 2).
Clicked button 'addresses_next_button' (attempt 3).
Target element 'ell_next_button' found after 3 attempts.
Clicked button 'ell_next_button' (attempt 1).
Target element 'sped_next_button' found after 1 attempts.
Clicked button 'sped_next_button' (attempt 1).
Target element 'home_schools_list' found after 1 attempts.
Success for McKay K-8 School
Success for Adams Elementary School
Success for East Boston Early Education Center
Success for Alighieri Dante Montessori School
Success for Otis Elementary School
Success for Mario Umana Academy
Success for O'Donnell Elementary School
Success for Kennedy Patrick J Elementary School
Success fo

In [10]:
print(school_list[0])

school_name='McKay K-8 School' address='122 Cottage St East Boston MA 02128' school_website='http://bostonpublicschools.org/Page/628' email='mckay@bostonpublicschools.org' distance_from_home='0.145 mi from home' school_hours='8:30am - 3:10pm Doors open for breakfast at 7:45.' preview_dates='(P) In-Person Session; (V) - Virtual Session\n• 11/14/2024, 10:00 AM - 11:00 AM, (P)\n• 12/11/2024, 1:30 PM - 2:00 PM, (P)\n• 1/7/2025, 2:00 PM - 3:00 PM, (P)' surround_care='After: Boston Scores Soccer and Writing Program; Harlem Lacrosse; Disney Musicals in Schools; YMCA Afterschool Programming; and Teacher-led clubs and programs based on student interest' school_description='The McKay K-8 School is a welcoming school community that embraces the uniqueness of every child. We empower students and we support their development by setting high expectations, actively involving families, and fostering core values of kindness, respect, effort, and responsibility.' grades_offered='K1 - 8' demand_reports='